# Explainable AI-Based Network Threat Detection and Classification
## Component 1: Dataset Handling, Preprocessing, and Exploratory Data Analysis (EDA)

**Dataset:** Canadian Institute for Cybersecurity CIC-IDS2017 Dataset  
**Author:** Senior Machine Learning Engineer (Data Engineering Lead)  

---
### Objectives
1. Inspect real CSV schemas, column names, character encodings, and data types.
2. Detect and handle missing values (`NaN`), infinite values (`Inf`), and anomalies (e.g. negative flow durations).
3. Perform deduplication to eliminate data leakage across evaluation splits.
4. Generate empirical visualizations of class imbalance, feature distributions, and collinearity.
5. Execute data-driven feature selection strictly on training data.
6. Provide a reusable, leak-free preprocessing pipeline artifact for Members 2, 3, and 4.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.dataset_config import RAW_DATA_DIR, FIGURES_DIR, SCHEMAS_DIR, MODELS_DIR
from src.dataset.loader import DatasetLoader
from src.dataset.schema import (
    CANONICAL_COLUMNS,
    TARGET_COLUMN,
    CONSTANT_COLUMNS,
    DUPLICATE_COLUMNS,
    normalize_label
)
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.splitter import DataSplitter
from src.preprocessing.pipeline import PreprocessingPipeline
from src.feature_selection.selector import FeatureSelector
from src.eda.visualizer import EDAVisualizer

print('All modules loaded successfully!')

## 1. Inspecting Dataset Files and Schemas
We inspect the 8 CSV files provided by CIC-IDS2017.

In [ ]:
loader = DatasetLoader(RAW_DATA_DIR)
csv_files = loader.get_csv_files()
print(f'Discovered {len(csv_files)} CSV files in {RAW_DATA_DIR}:')
for f in csv_files:
    print(f' - {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)')

## 2. Loading Balanced Sample for Exploratory Analysis

In [ ]:
# Load sample of 15,000 rows per file for fast interactive exploration
sample_df = loader.load_combined(nrows_per_file=15000, normalize_labels=True)
print(f'Loaded sample shape: {sample_df.shape}')
sample_df.head(3)

## 3. Ground-Truth Class Imbalance & Threat Families

In [ ]:
visualizer = EDAVisualizer(FIGURES_DIR)
# Plot class distribution (log scale to see minority classes)
fig_path = visualizer.plot_class_distribution(sample_df)
from IPython.display import Image, display
display(Image(filename=str(fig_path)))

## 4. Missing Values & Infinite Values Anomaly Analysis

In [ ]:
# Count NaNs and Infs
nans = sample_df.isna().sum()
cols_with_nan = nans[nans > 0].to_dict()
num_cols = sample_df.select_dtypes(include=[np.number]).columns
infs = {}
for c in num_cols:
    cnt = int(np.isinf(sample_df[c]).sum())
    if cnt > 0:
        infs[c] = cnt

print('Columns with NaNs:', cols_with_nan)
print('Columns with Infs:', infs)

fig_path2 = visualizer.plot_missing_and_inf_analysis(cols_with_nan, infs, len(sample_df))
display(Image(filename=str(fig_path2)))

## 5. Feature Correlation Heatmap

In [ ]:
key_feats = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward'
]
fig_path3 = visualizer.plot_correlation_heatmap(sample_df, key_feats)
display(Image(filename=str(fig_path3)))

## 6. Preprocessing & Leakage-Free Stratified Splitting

In [ ]:
cleaner = DataCleaner()
cleaned_sample, audit = cleaner.clean_dataframe(sample_df, is_training=True)
print('Cleaning Audit:')
for k, v in audit.items():
    print(f'  {k}: {v}')

splitter = DataSplitter(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42)
train_df, val_df, test_df, split_report = splitter.split_dataframe(cleaned_sample)
print(f'Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')

## 7. Systematic Feature Selection & Validation Benchmarking

In [ ]:
X_train, y_train_str = cleaner.separate_features_and_target(train_df)
X_val, y_val_str = cleaner.separate_features_and_target(val_df)
y_train = (y_train_str != 'BENIGN').astype(int).values
y_val = (y_val_str != 'BENIGN').astype(int).values

# Fit imputer on train only
X_train_num = X_train.replace([np.inf, -np.inf], np.nan)
train_meds = X_train_num.median()
X_train_imp = X_train_num.fillna(train_meds)
X_val_imp = X_val.replace([np.inf, -np.inf], np.nan).fillna(train_meds)

selector = FeatureSelector(top_k=25, random_state=42)
selected_feats = selector.fit_and_select(X_train_imp, y_train, X_val_imp, y_val)

fig_path4 = visualizer.plot_rf_feature_importance(selector.feature_importances_, top_n=25)
display(Image(filename=str(fig_path4)))

## 8. Exporting Fitted Pipeline for Members 2, 3, and 4

In [ ]:
pipeline = PreprocessingPipeline(scaler_type='robust', selected_features=selected_feats)
pipeline.fit(X_train)
pipeline_path = pipeline.save(MODELS_DIR / 'preprocessor.joblib')

# Test transform on unseen validation data
X_val_trans = pipeline.transform(X_val)
print(f'Successfully transformed validation set to shape: {X_val_trans.shape}')
print(f'Preprocessor ready at: {pipeline_path}')